# Import Libraries and Create Dataset

In [37]:
import pandas as pd
import numpy as np

data = {
    'transaction_id': range(1, 21),
    'date': pd.date_range('2024-10-01', periods=20, freq='D'),
    'region': ['North', 'South', 'East', 'West', None, 'North', 'South', None, 'East', 'West',
               'North', 'South', 'East', 'West', 'North', None, 'East', 'West', 'North', 'South'],
    'product_category': ['Electronics', 'Clothing', None, 'Books', 'Electronics', 'Home', 
                         'Clothing', 'Books', 'Electronics', None, 'Home', 'Clothing', 
                         'Books', 'Electronics', 'Home', 'Clothing', 'Books', 'Electronics', 
                         'Home', 'Clothing'],
    'sales_amount': [1200, 450, 890, None, 1500, 670, None, 340, 2100, 780, 
                     560, None, 420, 1800, 920, 510, 380, None, 1100, 640],
    'quantity': [2, 5, 3, 1, None, 4, 2, 3, 1, 5, 
                 3, 2, None, 1, 4, 3, 2, 1, None, 4],
    'customer_age': [25, 34, None, 45, 29, None, 38, 52, 27, 41, 
                     33, None, 48, 26, 35, 42, None, 31, 39, 44],
    'payment_method': ['Credit Card', 'UPI', 'Cash', 'Debit Card', 'Credit Card', 
                       'UPI', 'Cash', None, 'Credit Card', 'UPI', 'Debit Card', 
                       'Cash', 'Credit Card', None, 'UPI', 'Cash', 'Debit Card', 
                       'Credit Card', 'UPI', None]
}

df = pd.DataFrame(data)
df.head()

,transaction_id,date,region,product_category,sales_amount,quantity,customer_age,payment_method
0,1,2024-10-01,North,Electronics,1200.0,2.0,25.0,Credit Card
1,2,2024-10-02,South,Clothing,450.0,5.0,34.0,UPI
2,3,2024-10-03,East,None,890.0,3.0,NaN,Cash
3,4,2024-10-04,West,Books,NaN,1.0,45.0,Debit Card
4,5,2024-10-05,None,Electronics,1500.0,NaN,29.0,Credit Card


# Task 1: Load and Inspect Data 
# DataFrame Information

In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    20 non-null     int64         
 1   date              20 non-null     datetime64[ns]
 2   region            17 non-null     object        
 3   product_category  18 non-null     object        
 4   sales_amount      16 non-null     float64       
 5   quantity          17 non-null     float64       
 6   customer_age      16 non-null     float64       
 7   payment_method    17 non-null     object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(3)
memory usage: 1.4+ KB


In [39]:
df.isna().sum()

transaction_id      0
date                0
region              3
product_category    2
sales_amount        4
quantity            3
customer_age        4
payment_method      3
dtype: int64

# Task 2: Handle Missing Values
# 1. Fill Region with Mode

In [40]:
df['region'] = df['region'].fillna(df['region'].mode()[0])


# 2. Fill Product Category with Mode

In [41]:
df['product_category'] = df['product_category'].fillna(df['product_category'].mode()[0])


# 3. Fill Sales Amount with Median


In [42]:
df['sales_amount'] = df['sales_amount'].fillna(df['sales_amount'].median())


# 4. Forward Fill Quantity


In [43]:
df['quantity'] = df['quantity'].ffill()


# 5. Fill Customer Age with Mean (Rounded)


In [44]:
df['customer_age'] = df['customer_age'].fillna(round(df['customer_age'].mean()))


# 6. Drop Rows with Missing Payment Method 


In [45]:
df = df.dropna(subset=['payment_method'])

# Verify No Missing Values


In [46]:
df.isna().sum()

transaction_id      0
date                0
region              0
product_category    0
sales_amount        0
quantity            0
customer_age        0
payment_method      0
dtype: int64

# Task 3: GroupBy Analysis
# Total Sales by Region

In [47]:
sales_by_region = df.groupby('region')['sales_amount'].sum()

print(sales_by_region)

region
East     3790.0
North    6460.0
South    1900.0
West     2230.0
Name: sales_amount, dtype: float64


# Average Sales by Product Category

In [48]:
avg_sales_category = df.groupby('product_category')['sales_amount'].mean()

print(avg_sales_category)

product_category
Books           508.333333
Clothing        680.000000
Electronics    1381.250000
Home            812.500000
Name: sales_amount, dtype: float64


# Sales and Quantity by Region and Product Category

In [49]:
region_product_sales = df.groupby(
    ['region', 'product_category']
)[['sales_amount', 'quantity']].sum().reset_index()

print(region_product_sales)

  region product_category  sales_amount  quantity
0   East            Books         800.0       4.0
1   East         Clothing         890.0       3.0
2   East      Electronics        2100.0       1.0
3  North         Clothing         510.0       3.0
4  North      Electronics        2700.0       3.0
5  North             Home        3250.0      12.0
6  South         Clothing        1900.0       9.0
7   West            Books         725.0       1.0
8   West         Clothing         780.0       5.0
9   West      Electronics         725.0       1.0


# Top 3 Region-Product Combinations by Sales

In [50]:
top3_sales = region_product_sales.sort_values(
    by='sales_amount',
    ascending=False
).head(3)

print(top3_sales)

  region product_category  sales_amount  quantity
5  North             Home        3250.0      12.0
4  North      Electronics        2700.0       3.0
2   East      Electronics        2100.0       1.0


# Task 4: Custom Aggregation
# Create Sales Range Function

In [51]:
def sales_range(series):
    return series.max() - series.min()

# Apply Sales Range by Region

In [52]:
range_by_region = df.groupby('region')['sales_amount'].apply(sales_range)

print(range_by_region)

region
East     1720.0
North     990.0
South     275.0
West       55.0
Name: sales_amount, dtype: float64


# Multiple Aggregations by Region

In [53]:
region_metrics = df.groupby('region').agg({
    'sales_amount': ['sum', 'mean', 'max'],
    'quantity': ['sum', 'min']
})

print(region_metrics)

       sales_amount                     quantity     
                sum        mean     max      sum  min
region                                               
East         3790.0  947.500000  2100.0      8.0  1.0
North        6460.0  922.857143  1500.0     18.0  1.0
South        1900.0  633.333333   725.0      9.0  2.0
West         2230.0  743.333333   780.0      7.0  1.0
